In [2]:
import pandas as pd, os

OUTPUT = '/workspaces/curling_data/output/'

events      = pd.read_csv(OUTPUT + 'events.csv')
matches     = pd.read_csv(OUTPUT + 'matches.csv')
ends        = pd.read_csv(OUTPUT + 'ends.csv')
result_urls = pd.read_csv(OUTPUT + 'result_urls.csv')

shots_meta = pd.read_csv(OUTPUT + 'shot_locations.csv',
                         usecols=['event_id','match_id','end_number','shot_number','turn'])

print(f'events: {len(events)}, matches: {len(matches)}, ends: {len(ends)}, shots: {len(shots_meta)}')

events: 260, matches: 4366, ends: 30550, shots: 488800


In [4]:
# ── ISSUE 1: 171 events with zero data ─────────────────────────────────────
# Step 1 — which event_ids have NO matches at all?
matched_events = set(matches['event_id'])
empty_events = events[~events['event_id'].isin(matched_events)].copy()
print(f'Events with zero matches: {len(empty_events)}')
print(f'Events WITH matches: {len(events) - len(empty_events)}')

# Year distribution
print('\nEmpty events by year:')
print(empty_events['year'].value_counts().sort_index())

Events with zero matches: 171
Events WITH matches: 89

Empty events by year:
year
2013     9
2014    11
2015    15
2016    16
2017    14
2018    10
2019    18
2020     5
2021     9
2022    16
2023    15
2024    16
2025    17
Name: count, dtype: int64


In [4]:
# Step 2 — cross-reference with result_urls.csv
# Normalise event names for rough matching
result_urls['norm_name'] = result_urls['tournament_name'].str.strip().str.lower()
events['norm_name'] = events['event_name'].str.strip().str.lower()
empty_events['norm_name'] = empty_events['event_name'].str.strip().str.lower()

# Exact-match on (norm_name, year)
url_keys = set(zip(result_urls['norm_name'], result_urls['year'].astype('Int64')))
empty_events['year_int'] = pd.to_numeric(empty_events['year'], errors='coerce').astype('Int64')
empty_events['in_result_urls'] = empty_events.apply(
    lambda r: (r['norm_name'], r['year_int']) in url_keys, axis=1
)

print('Empty events cross-referenced with result_urls:')
print(empty_events.groupby(['in_result_urls', 'year_int']).size().reset_index(name='count').to_string(index=False))

print(f"\nOf {len(empty_events)} empty events:")
print(f"  Have a result_urls entry: {empty_events['in_result_urls'].sum()}")
print(f"  NOT in result_urls:       {(~empty_events['in_result_urls']).sum()}")

Empty events cross-referenced with result_urls:
 in_result_urls  year_int  count
           True      2013      9
           True      2014     11
           True      2015     15
           True      2016     16
           True      2017     14
           True      2018     10
           True      2019     18
           True      2020      5
           True      2021      9
           True      2022     16
           True      2023     15
           True      2024     16
           True      2025     17

Of 171 empty events:
  Have a result_urls entry: 171
  NOT in result_urls:       0


In [5]:
# Step 3 — for events NOT in result_urls, check if they have a pdf_file listed
# (scraper may have tried to process them from a direct PDF path)
print('Empty events with pdf_file set:')
has_pdf = empty_events[empty_events.get('pdf_file', pd.Series(dtype=str)).notna() 
                       if 'pdf_file' in empty_events.columns else pd.Series(False, index=empty_events.index)]

print(empty_events[['event_id','event_name','year','pdf_file']].head(30).to_string(index=False) 
      if 'pdf_file' in empty_events.columns else 'No pdf_file column')

print('\nAll columns in events.csv:', events.columns.tolist())
print('\nSample empty-event rows:')
print(empty_events[['event_id','event_name','year']].head(30).to_string(index=False))

Empty events with pdf_file set:
 event_id                                            event_name  year                                   pdf_file
        5          World Mixed Doubles Qualification Event 2025  2025                  WMDQE2025_ResultsBook.pdf
        6             World Junior-B Curling Championships 2025  2025              WJBCC2025_ResultsBook_Men.pdf
        7             World Junior-B Curling Championships 2025  2025        WJBCC2025_ResultsBook_Women.pdf?v=2
       13        European Curling Championships 2025 B-Division  2025 ECC2025_ResultsBook_Men_B-Division.pdf?V=2
       14        European Curling Championships 2025 B-Division  2025   ECC2025_ResultsBook_Women_B-Division.pdf
       17 Pan Continental Curling Championships 2025 B-Division  2025    PCCC2025_ResultsBook_Men_B-Division.pdf
       18 Pan Continental Curling Championships 2025 B-Division  2025  PCCC2025_ResultsBook_Women_B-Division.pdf
       19                  Pre-Olympic Qualification Event 2025 

In [6]:
# Step 4 — event type / name pattern analysis
# Extract the tournament category from the event name
def classify_event(name):
    name = str(name).lower()
    if 'mixed doubles' in name:        return 'Mixed Doubles'
    if 'senior' in name:               return 'Senior'
    if 'wheelchair' in name:           return 'Wheelchair'
    if 'junior-b' in name or 'jb' in name: return 'Junior-B'
    if 'junior' in name:               return 'Junior'
    if 'olympic' in name:              return 'Olympic'
    if 'paralympic' in name:           return 'Paralympic'
    if 'universiade' in name:          return 'Universiade'
    if 'youth' in name:                return 'Youth'
    if 'world cup' in name or 'cwc' in name: return 'World Cup'
    if 'b-division' in name or 'c-division' in name: return 'B/C Division'
    if 'qualification' in name:        return 'Qualification'
    if 'mixed curling' in name:        return 'Mixed 4-person'
    if 'asian' in name:                return 'Asian Games'
    return 'Other'

empty_events['category'] = empty_events['event_name'].apply(classify_event)

print('Empty events by category:')
print(empty_events.groupby('category').agg(
    count=('event_id','count'),
    years=('year', lambda x: f"{int(x.min())}-{int(x.max())}" if x.notna().any() else 'N/A')
).sort_values('count', ascending=False).to_string())

Empty events by category:
                count      years
category                        
B/C Division       48  2013-2025
Other              20  2013-2025
Senior             18  2014-2025
Junior-B           16  2017-2025
Wheelchair         15  2013-2025
Mixed Doubles      14  2014-2025
Olympic            12  2013-2025
Junior              9  2013-2017
Mixed 4-person      8  2015-2024
Qualification       6  2019-2022
Universiade         4  2013-2019
Paralympic          1  2014-2014


In [7]:
# ── ISSUE 2: 867 matches with metadata but no ends/shots ───────────────────
# Identify which matches have NO ends
matches_with_ends_set = set(zip(ends['event_id'], ends['match_id']))
matches['has_ends'] = matches.apply(
    lambda r: (r['event_id'], r['match_id']) in matches_with_ends_set, axis=1
)
no_ends = matches[~matches['has_ends']].copy()
print(f'Matches with no ends: {len(no_ends)}')

# Join to events for names
no_ends_ev = no_ends.merge(events[['event_id','event_name','year']], on='event_id', how='left')
print('\nBreakdown by event (top 20):')
print(no_ends_ev.groupby(['event_id','year','event_name']).size()
      .reset_index(name='missing_matches')
      .sort_values('missing_matches', ascending=False)
      .head(20).to_string(index=False))

Matches with no ends: 867

Breakdown by event (top 20):
 event_id  year                                           event_name  missing_matches
      126  2021        World Mixed Doubles Curling Championship 2021               99
       24  2025        World Mixed Doubles Curling Championship 2025               99
       75  2023        World Mixed Doubles Curling Championship 2023               98
       50  2024        World Mixed Doubles Curling Championship 2024               98
      102  2022        World Mixed Doubles Curling Championship 2022               98
       10  2025                     Olympic Qualification Event 2025               53
      108  2022                            Olympic Winter Games 2022               49
        4  2026    Milano Cortina 2026 Olympic Winter Games, Curling               49
      185  2018                            Olympic Winter Games 2018               33
        1  2026 Milano Cortina 2026 Paralympic Winter Games, Curling               3

In [8]:
# Sub-classify: WMD vs Olympic/other
no_ends_ev['category'] = no_ends_ev['event_name'].apply(classify_event)

print('No-ends matches by category:')
print(no_ends_ev.groupby('category').agg(
    matches=('match_id','count'),
    events=('event_id','nunique'),
    years=('year', lambda x: f"{int(x.min())}-{int(x.max())}" if x.notna().any() else 'N/A')
).sort_values('matches', ascending=False).to_string())

No-ends matches by category:
               matches  events      years
category                                 
Mixed Doubles      535       7  2018-2025
Olympic            204       5  2018-2026
World Cup           78       4  2018-2018
Paralympic          32       1  2026-2026
Asian Games         18       1  2025-2025


In [9]:
# 2b — Olympic/Paralympic split: which ROUNDS are missing ends?
olympic_no_ends = no_ends_ev[no_ends_ev['category'].isin(['Olympic','Paralympic'])]
olympic_has_ends = matches[matches['has_ends']].merge(
    events[['event_id','event_name','year']], on='event_id', how='left'
)
olympic_has_ends['category'] = olympic_has_ends['event_name'].apply(classify_event)
olympic_has_ends = olympic_has_ends[olympic_has_ends['category'].isin(['Olympic','Paralympic'])]

print('ROUND breakdown — Olympic/Paralympic WITH ends:')
print(olympic_has_ends['round'].value_counts().head(15))

print('\nROUND breakdown — Olympic/Paralympic WITHOUT ends:')
print(olympic_no_ends['round'].value_counts().head(15))

ROUND breakdown — Olympic/Paralympic WITH ends:
round
Round Robin Session 5 - Sheet A    15
Round Robin Session 4 - Sheet B    15
Round Robin Session 7 - Sheet B    14
Round Robin Session 6 - Sheet A    14
Round Robin Session 4 - Sheet D    14
Round Robin Session 3 - Sheet B    14
Round Robin Session 3 - Sheet C    14
Round Robin Session 1 - Sheet A    14
Round Robin Session 1 - Sheet C    14
Round Robin Session 7 - Sheet D    13
Round Robin Session 6 - Sheet C    13
Round Robin Session 6 - Sheet D    13
Round Robin Session 5 - Sheet C    13
Round Robin Session 2 - Sheet A    13
Round Robin Session 2 - Sheet B    13
Name: count, dtype: int64

ROUND breakdown — Olympic/Paralympic WITHOUT ends:
round
Round Robin Session 7              9
Round Robin Session 3              9
Round Robin Session 1              9
Round Robin Session 6              8
Round Robin Session 5              8
Round Robin Session 2              8
Round Robin Session 4              7
Round Robin Session 7 - Sheet C  

In [10]:
# 2b — per-event breakdown: how many matches with/without ends per Olympic event
olympic_events = events[events['event_name'].apply(classify_event).isin(['Olympic','Paralympic'])]
print('Olympic/Paralympic events:')
for _, ev_row in olympic_events.iterrows():
    eid = ev_row['event_id']
    ev_matches = matches[matches['event_id'] == eid]
    with_ends = ev_matches['has_ends'].sum()
    without_ends = (~ev_matches['has_ends']).sum()
    print(f"  [{eid}] {ev_row['event_name']} {int(ev_row['year']) if pd.notna(ev_row['year']) else ''}: "
          f"{with_ends} with ends, {without_ends} without")

Olympic/Paralympic events:
  [1] Milano Cortina 2026 Paralympic Winter Games, Curling 2026: 49 with ends, 32 without
  [4] Milano Cortina 2026 Olympic Winter Games, Curling 2026: 98 with ends, 49 without
  [8] Olympic Qualification Event 2025 2025: 30 with ends, 0 without
  [9] Olympic Qualification Event 2025 2025: 30 with ends, 0 without
  [10] Olympic Qualification Event 2025 2025: 0 with ends, 53 without
  [19] Pre-Olympic Qualification Event 2025 2025: 0 with ends, 0 without
  [20] Pre-Olympic Qualification Event 2025 2025: 0 with ends, 0 without
  [59] Winter Youth Olympic Games 2024 2024: 0 with ends, 0 without
  [107] Paralympic Winter Games 2022 2022: 59 with ends, 0 without
  [108] Olympic Winter Games 2022 2022: 98 with ends, 49 without
  [111] Olympic Qualification Event 2021 2021: 0 with ends, 20 without
  [112] Olympic Qualification Event 2021 2021: 17 with ends, 0 without
  [113] Olympic Qualification Event 2021 2021: 11 with ends, 0 without
  [121] Pre-Olympic Qualifica

In [11]:
# 2b — Investigate missing-end rounds more carefully for the 2026 Olympics (event_id=4)
oly2026 = matches[matches['event_id'] == 4].copy()
oly2026_no_ends = oly2026[~oly2026['has_ends']]
oly2026_has_ends = oly2026[oly2026['has_ends']]

print('2026 OWG — WITH ends:')
print(oly2026_has_ends['round'].value_counts())
print(f'\n2026 OWG — WITHOUT ends:')
print(oly2026_no_ends[['match_id','round','team1_code','team2_code']].to_string(index=False))

2026 OWG — WITH ends:
round
Gold Medal Game                     2
Bronze Medal Game                   2
Semi-final - Sheet B                2
Semi-final - Sheet D                2
Round Robin Session 12 - Sheet A    2
Round Robin Session 12 - Sheet B    2
Round Robin Session 12 - Sheet C    2
Round Robin Session 12 - Sheet D    2
Round Robin Session 11 - Sheet A    2
Round Robin Session 11 - Sheet B    2
Round Robin Session 11 - Sheet C    2
Round Robin Session 11 - Sheet D    2
Round Robin Session 10 - Sheet A    2
Round Robin Session 10 - Sheet B    2
Round Robin Session 10 - Sheet C    2
Round Robin Session 9 - Sheet B     2
Round Robin Session 9 - Sheet C     2
Round Robin Session 9 - Sheet D     2
Round Robin Session 8 - Sheet A     2
Round Robin Session 8 - Sheet B     2
Round Robin Session 8 - Sheet C     2
Round Robin Session 8 - Sheet D     2
Round Robin Session 7 - Sheet B     2
Round Robin Session 7 - Sheet C     2
Round Robin Session 7 - Sheet D     2
Round Robin Session 6 

In [12]:
# Check if missing-end Olympic matches share date/start_time patterns
# that differ from matches with ends (hint at different page format)
print('Dates in 2026 OWG matches WITHOUT ends:')
print(oly2026_no_ends[['match_id','date','round','start_time']].to_string(index=False))

print('\nSample dates in 2026 OWG matches WITH ends:')
print(oly2026_has_ends[['match_id','date','round','start_time']].head(10).to_string(index=False))

Dates in 2026 OWG matches WITHOUT ends:
 match_id            date                            round                                                                   start_time
       99 TUE 10 FEB 2026                  Gold Medal Game                                 18:05 Partita per la medaglia d'oro / Finale
      100 TUE 10 FEB 2026                Bronze Medal Game   14:05 Partita per la medaglia di bronzo / Match pour la médaille de bronze
      101  MON 9 FEB 2026             Semi-final - Sheet B       18:05 Semifinale - Campo B - piste B / Demi-finale - piste B - Sheet B
      102  MON 9 FEB 2026             Semi-final - Sheet D       18:05 Semifinale - Campo D - piste D / Demi-finale - piste D - Sheet D
      103  MON 9 FEB 2026 Round Robin Session 13 - Sheet A 10:05 Girone Sessione 13 - Campo A / Tour préliminaire, session 13 - piste A
      104  MON 9 FEB 2026 Round Robin Session 13 - Sheet B 10:05 Girone Sessione 13 - Campo B / Tour préliminaire, session 13 - piste B
      10

In [13]:
# 2a — World Mixed Doubles: are match counts consistent with expected tournament size?
# Typical WMD: 36 teams, round robin + playoff ~ 120+ games
wmd_events = events[events['event_name'].str.contains('Mixed Doubles', case=False, na=False)]
print('WMD events:')
for _, ev_row in wmd_events.iterrows():
    eid = ev_row['event_id']
    ev_m = matches[matches['event_id'] == eid]
    with_e = ev_m['has_ends'].sum() if len(ev_m) else 0
    without_e = (~ev_m['has_ends']).sum() if len(ev_m) else 0
    print(f"  [{eid}] {ev_row['event_name']} {int(ev_row['year']) if pd.notna(ev_row['year']) else ''}: "
          f"{len(ev_m)} total matches, {with_e} with ends, {without_e} without")

WMD events:
  [5] World Mixed Doubles Qualification Event 2025 2025: 0 total matches, 0 with ends, 0 without
  [21] World Junior Mixed Doubles Curling Championship 2025 2025: 0 total matches, 0 with ends, 0 without
  [24] World Mixed Doubles Curling Championship 2025 2025: 99 total matches, 0 with ends, 99 without
  [31] World Wheelchair Mixed Doubles Championship 2025 2025: 0 total matches, 0 with ends, 0 without
  [37] World Mixed Doubles Qualification Event 2024 2024: 0 total matches, 0 with ends, 0 without
  [50] World Mixed Doubles Curling Championship 2024 2024: 98 total matches, 0 with ends, 98 without
  [55] World Wheelchair Mixed Doubles Championship 2024 2024: 0 total matches, 0 with ends, 0 without
  [62] World Mixed Doubles Qualification Event 2023 2023: 0 total matches, 0 with ends, 0 without
  [75] World Mixed Doubles Curling Championship 2023 2023: 98 total matches, 0 with ends, 98 without
  [81] World Wheelchair Mixed Doubles Curling Championships 2023 2023: 0 total mat

In [14]:
# ── ISSUE 6: 28% of ends missing time_left ─────────────────────────────────
ends['has_time'] = ends['team1_time_left'].notna() & ends['team2_time_left'].notna()

# Per-event null rate
time_stats = (
    ends.groupby('event_id')
    .agg(total_ends=('end_number','count'),
         ends_with_time=('has_time','sum'))
    .reset_index()
)
time_stats['pct_with_time'] = (time_stats['ends_with_time'] / time_stats['total_ends'] * 100).round(1)
time_stats = time_stats.merge(events[['event_id','event_name','year']], on='event_id', how='left')
time_stats['has_time_data'] = time_stats['pct_with_time'] > 0

print(f"Events with ANY time data:  {time_stats['has_time_data'].sum()}")
print(f"Events with ZERO time data: {(~time_stats['has_time_data']).sum()}")

# Partial time coverage (>0% but <100%)
partial = time_stats[(time_stats['pct_with_time'] > 0) & (time_stats['pct_with_time'] < 100)]
print(f"\nEvents with partial time coverage: {len(partial)}")
if len(partial):
    print(partial[['event_id','year','event_name','total_ends','pct_with_time']].to_string(index=False))

Events with ANY time data:  74
Events with ZERO time data: 6

Events with partial time coverage: 27
 event_id  year                                     event_name  total_ends  pct_with_time
       11  2025 European Curling Championships 2025 A-Division         437           92.2
       12  2025 European Curling Championships 2025 A-Division         444           93.5
       29  2025          World Men's Curling Championship 2025         730           99.9
       39  2024 European Curling Championships 2024 A-Division         439           94.3
       54  2024        World Women's Curling Championship 2024         742           82.6
       78  2023          World Men's Curling Championship 2023         717           96.1
      105  2022          World Men's Curling Championship 2022         751            9.5
      106  2022        World Women's Curling Championship 2022         662           10.3
      112  2021               Olympic Qualification Event 2021         157            8.3


In [15]:
# Year-by-year time-data coverage
yearly = time_stats.groupby('year').agg(
    total_events=('event_id','count'),
    events_with_time=('has_time_data','sum')
).reset_index()
yearly['pct_events_with_time'] = (yearly['events_with_time'] / yearly['total_events'] * 100).round(0).astype(int)
print('Time-data coverage by year:')
print(yearly.to_string(index=False))

Time-data coverage by year:
 year  total_events  events_with_time  pct_events_with_time
 2017             8                 8                   100
 2018            14                10                    71
 2019             8                 8                   100
 2020             2                 2                   100
 2021             7                 7                   100
 2022            10                10                   100
 2023             8                 8                   100
 2024             8                 8                   100
 2025            11                11                   100
 2026             4                 2                    50


In [16]:
# Add has_time_data flag to events.csv
events_full = pd.read_csv(OUTPUT + 'events.csv')

# For events with no ends rows, flag = False
time_flag = time_stats[['event_id','has_time_data']].copy()
time_flag.rename(columns={'has_time_data': 'has_time_data'}, inplace=True)

events_full = events_full.merge(time_flag, on='event_id', how='left')
events_full['has_time_data'] = events_full['has_time_data'].fillna(False).astype(bool)

events_full.to_csv(OUTPUT + 'events.csv', index=False)
print(f"events.csv updated with has_time_data column.")
print(f"  True (has time data):  {events_full['has_time_data'].sum()}")
print(f"  False (no time data): {(~events_full['has_time_data']).sum()}")
print(f"\nNew columns: {events_full.columns.tolist()}")

events.csv updated with has_time_data column.
  True (has time data):  74
  False (no time data): 186

New columns: ['event_id', 'event_name', 'year', 'location', 'gender', 'pdf_file', 'has_time_data']


In [17]:
# ── ISSUE 7: 7 shots with NULL turn ────────────────────────────────────────
null_turn = shots_meta[shots_meta['turn'].isna()].copy()
print(f'Shots with NULL turn: {len(null_turn)}')

null_turn_ev = null_turn.merge(events[['event_id','event_name','year']], on='event_id', how='left')
null_turn_mat = null_turn_ev.merge(
    matches[['event_id','match_id','team1_code','team2_code']], on=['event_id','match_id'], how='left'
)
print('\nNULL-turn shots:')
print(null_turn_mat[['event_id','match_id','end_number','shot_number',
                      'team1_code','team2_code','year','event_name']].to_string(index=False))

Shots with NULL turn: 7

NULL-turn shots:
 event_id  match_id  end_number  shot_number team1_code team2_code  year                                            event_name
       11        40           5           16        SCO        CZE  2025        European Curling Championships 2025 A-Division
       15         8           8            1        KOR        PHI  2025 Pan Continental Curling Championships 2025 A-Division
       15        12           8            4        USA        CHN  2025 Pan Continental Curling Championships 2025 A-Division
       16        16           4            1        NZL        KOR  2025 Pan Continental Curling Championships 2025 A-Division
       16        20           7            4        JPN        USA  2025 Pan Continental Curling Championships 2025 A-Division
       39        28           8           16        TUR        EST  2024        European Curling Championships 2024 A-Division
      120        65           8            3        CAN        USA  2

In [18]:
# Check turn value distribution overall (sanity check)
shots_meta_full = pd.read_csv(OUTPUT + 'shot_locations.csv',
                              usecols=['event_id','match_id','end_number','shot_number','turn','shot_type','accuracy'])
print('Overall turn value distribution:')
print(shots_meta_full['turn'].value_counts(dropna=False))

# What shot_type/accuracy do the NULL-turn shots have?
null_turn_full = shots_meta_full[
    shots_meta_full.apply(lambda r: (r['event_id'], r['match_id'], r['end_number'], r['shot_number'])
                          in set(zip(null_turn['event_id'], null_turn['match_id'],
                                     null_turn['end_number'], null_turn['shot_number'])), axis=1)
]
print('\nDetails of NULL-turn shots:')
print(null_turn_full[['event_id','match_id','end_number','shot_number',
                       'shot_type','accuracy','turn']].to_string(index=False))

Overall turn value distribution:
turn
Counter-clockwise    242808
Clockwise            239383
Not considered         6602
NaN                       7
Name: count, dtype: int64

Details of NULL-turn shots:
 event_id  match_id  end_number  shot_number                         shot_type  accuracy turn
       11        40           5           16        Through Hog line violation       0.0  NaN
       15         8           8            1        Through Hog line violation       0.0  NaN
       15        12           8            4 Through Free Guard Zone violation       0.0  NaN
       16        16           4            1              Through Burned stone       0.0  NaN
       16        20           7            4 Through Free Guard Zone violation       0.0  NaN
       39        28           8           16                           Through     100.0  NaN
      120        65           8            3                           Through       0.0  NaN


In [19]:
# Check if NULL turn is associated with any specific shot_type (e.g. timeout)
print('shot_type for NULL-turn shots:')
print(null_turn_full['shot_type'].value_counts(dropna=False))

# Compare shot numbers — are these odd or even (determines which team threw)?
print('\nshot_number for NULL-turn shots (odd=team1, even=team2):')
print(null_turn_full['shot_number'].value_counts())

# Check if accuracy is also NULL for these shots (would indicate whole-row parse failure)
both_null = null_turn_full[null_turn_full['accuracy'].isna() | (null_turn_full['accuracy'] == '')]
print(f'\nNULL-turn shots that ALSO have NULL/empty accuracy: {len(both_null)}')

shot_type for NULL-turn shots:
shot_type
Through Hog line violation           2
Through Free Guard Zone violation    2
Through                              2
Through Burned stone                 1
Name: count, dtype: int64

shot_number for NULL-turn shots (odd=team1, even=team2):
shot_number
16    2
1     2
4     2
3     1
Name: count, dtype: int64

NULL-turn shots that ALSO have NULL/empty accuracy: 0


In [3]:
# ── ISSUE 7 FIX: patch 7 NULL-turn "Through" rows in shot_locations.csv ────
shots_full = pd.read_csv(OUTPUT + 'shot_locations.csv')
print(f'Before patch — NULL turn count: {shots_full["turn"].isna().sum()}')

null_mask = shots_full['turn'].isna() & shots_full['shot_type'].str.startswith('Through', na=False)
shots_full.loc[null_mask, 'turn'] = 'Not considered'

print(f'After patch  — NULL turn count: {shots_full["turn"].isna().sum()}')
print(f'Rows patched: {null_mask.sum()}')

shots_full.to_csv(OUTPUT + 'shot_locations.csv', index=False)
print('shot_locations.csv saved.')


Before patch — NULL turn count: 7
After patch  — NULL turn count: 0
Rows patched: 7
shot_locations.csv saved.


In [5]:
# ── ISSUE 2 deep dive: WMD matches — shot image count analysis ──────────────
# The Issue 3 fix records end scores when len(shot_images) != 16.
# For WMD (6 stones/team, 12 shots/end), ALL ends have 12 images → zero ends currently.
# After re-scraping with the fix, WMD matches will have end scores.
#
# Estimate: if ALL 535 WMD no-ends matches have ~5 ends each, that's ~2,675 new end rows.
# Olympic Mixed Doubles (204 matches × ~5 ends) = ~1,020 more end rows.
#
# Check: do any WMD events currently have partial ends data (to validate this hypothesis)?
wmd_event_ids = events[events['event_name'].str.contains('Mixed Doubles', case=False, na=False)]['event_id'].tolist()
olympic_event_ids = events[events['event_name'].str.contains('Olympic Winter Games|Paralympic Winter Games', 
                                                               case=False, na=False)]['event_id'].tolist()

print('WMD-category events with ANY ends data:')
wmd_with_ends = ends[ends['event_id'].isin(wmd_event_ids)]
print(f'  WMD ends rows: {len(wmd_with_ends)}  (from {wmd_with_ends["event_id"].nunique()} events)')

print('\nOlympic/Paralympic events with ends count:')
oly_ends = ends[ends['event_id'].isin(olympic_event_ids)]
print(f'  Olympic/Para ends rows: {len(oly_ends)}  (from {oly_ends["event_id"].nunique()} events)')

# Round types present in Olympic ends
print('\nTop round names in Olympic ends (to compare against no-ends matches):')
oly_ends_rounds = matches[matches['event_id'].isin(olympic_event_ids)].merge(
    ends[['event_id','match_id']].drop_duplicates(), on=['event_id','match_id'], how='inner'
)
print(oly_ends_rounds['round'].value_counts().head(10))


WMD-category events with ANY ends data:
  WMD ends rows: 0  (from 0 events)

Olympic/Paralympic events with ends count:
  Olympic/Para ends rows: 4039  (from 6 events)

Top round names in Olympic ends (to compare against no-ends matches):
round
Round Robin Session 12 - Sheet B    9
Round Robin Session 12 - Sheet C    9
Round Robin Session 12 - Sheet D    9
Round Robin Session 11 - Sheet A    9
Round Robin Session 11 - Sheet B    9
Round Robin Session 11 - Sheet C    9
Round Robin Session 10 - Sheet A    9
Round Robin Session 10 - Sheet B    9
Round Robin Session 10 - Sheet C    9
Round Robin Session 9 - Sheet B     9
Name: count, dtype: int64


In [6]:
# ── ISSUE 7: sync shot_locations.parquet with the already-patched CSV ────────
import pandas as pd

shots_parquet = pd.read_parquet(OUTPUT + 'shot_locations.parquet')
print(f'Before patch — NULL turn in parquet: {shots_parquet["turn"].isna().sum()}')

null_through = shots_parquet['turn'].isna() & shots_parquet['shot_type'].str.startswith('Through', na=False)
shots_parquet.loc[null_through, 'turn'] = 'Not considered'

print(f'After patch  — NULL turn in parquet: {shots_parquet["turn"].isna().sum()}')
print(f'Rows patched: {null_through.sum()}')

shots_parquet.to_parquet(OUTPUT + 'shot_locations.parquet', index=False)
print('shot_locations.parquet saved.')


Before patch — NULL turn in parquet: 7
After patch  — NULL turn in parquet: 0
Rows patched: 7
shot_locations.parquet saved.


In [11]:
import subprocess, shutil

pytest_exe = shutil.which('pytest') or 'pytest'
result = subprocess.run(
    [pytest_exe, 'tests/test_extract_shot_data.py', '--tb=short',
     '-k', 'not TestWithPDF and not TestMultiEvent'],
    capture_output=True, text=True
)
# Print only the summary line
for line in result.stdout.split('\n'):
    if 'passed' in line or 'failed' in line or 'error' in line.lower():
        print(line)
if result.returncode != 0:
    print(result.stdout[-2000:])


======================= 24 passed, 8 deselected in 1.46s =======================
